# Complete Exploratory Data Analysis

## Georgia Tech MSA Spring 2026 Practicum

This is where your full EDA goes. We look forward to digging deeper into your analysis here.

Read the [eda_outline.md](eda_outline.md) for more details.

In [ ]:
import polars as pl
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output
from eda_utils import load_eda_data, get_seasonal_metrics, plot_comparative_index


DATA_DIR = Path("data/Statsbomb")

In [2]:
matches_lf, events_lf = load_eda_data(DATA_DIR)
ref_matches = matches_lf.collect()
leagues = ref_matches["competition_name"].unique().to_list()
seasons = ref_matches["season_name"].unique().to_list()

In [ ]:
import polars as pl
import ipywidgets as widgets
from IPython.display import display, clear_output
# Ensure your utils are imported
from eda_utils import get_seasonal_metrics, plot_comparative_index

output = widgets.Output()


leagues = sorted(ref_matches["competition_name"].unique().to_list())
seasons = sorted(ref_matches["season_name"].unique().to_list())

dropdown_league = widgets.Dropdown(options=leagues, description='League:')
dropdown_season = widgets.Dropdown(options=seasons, description='Season:')
dropdown_team = widgets.Dropdown(description='Team:')
dropdown_metric = widgets.Dropdown(
    options=[('Expected Goals (xG)', 'total_xg'), ('Pass Volume', 'pass_count')],
    description='Metric:'
)


def update_teams(*args):
    """Updates team list based on League/Season selection."""
    filtered = ref_matches.filter(
        (pl.col("competition_name") == dropdown_league.value) &
        (pl.col("season_name") == dropdown_season.value)
    )
    teams = sorted(list(set(
        filtered["home_team"].to_list() + filtered["away_team"].to_list()
    )))
    dropdown_team.options = teams
    if teams:
        dropdown_team.value = teams[0]
# Attach observers
dropdown_league.observe(update_teams, 'value')
dropdown_season.observe(update_teams, 'value')


def run_eda(b):
    with output:
        clear_output(wait=True) # wait=True prevents flickering
        if not dropdown_team.value:
            print("Please select a team first.")
            return

        print(f"Analyzing {dropdown_team.value}...")
        try:

            df = get_seasonal_metrics(matches_lf, events_lf, dropdown_team.value)


            # .to_list() ensures we don't pass a "List of Lists" to Polars
            valid_ids = (
                ref_matches.filter(
                    (pl.col("competition_name") == dropdown_league.value) &
                    (pl.col("season_name") == dropdown_season.value)
                )
                .select("match_id")
                .unique()
                .to_series()
                .to_list()
            )


            display_df = df.filter(pl.col("match_id").is_in(valid_ids))

            if display_df.height == 0:
                print(f"No match events found for {dropdown_team.value} in this specific season.")
                return


            fig = plot_comparative_index(display_df, dropdown_team.value, dropdown_metric.value)
            fig.show()

        except Exception as e:
            print(f"Error during EDA: {e}")


btn_run = widgets.Button(description="Generate Index", button_style='success')
btn_run.on_click(run_eda)

# Initialize the team dropdown
update_teams()

# Display everything
display(widgets.VBox([
    widgets.HTML("<b>Step 1: Filter Dataset</b>"),
    widgets.HBox([dropdown_league, dropdown_season]),
    widgets.HTML("<b>Step 2: Select Team and Metric</b>"),
    widgets.HBox([dropdown_team, dropdown_metric]),
    btn_run,
    output
]))